# 00 · Setup and environment check (checkpoint CP0)

Run every cell from top to bottom in a **fresh** Colab runtime (CPU is enough; this uses no GPU units).

It mounts Google Drive, clones the `Paper1` repository, installs it, runs the test suite and the
repository audit, and saves the result to `Paper 1/runs/checkpoints/cp0_colab.json` on Drive.

Before running: in the Colab **Secrets** panel (key icon) add `GITHUB_TOKEN` (a read-only token) and
enable notebook access. While the repository is public the clone also works without it.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: no Drive, no clone
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        # drop the token from the stored remote URL
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[dev]'], check=True)

sys.path.insert(0, os.path.join(REPO, 'src'))        # makes the fresh install importable now
print('repo:', REPO, '| dry run:', DRY)

In [ ]:
# Cell 2: what this session runs on (no secrets are read or shown)
import json, logging
logging.basicConfig(level=logging.INFO, force=True)
from eviground.config import load_config
from eviground.env import get_paths, session_info

cfg = load_config('base')
info = session_info(cu_per_hour=float(cfg.compute.cu_per_hour))
print(json.dumps(info, indent=2))

In [ ]:
# Cell 3: test suite (in a dry run only the fast environment tests, to avoid running everything twice)
target = os.path.join(REPO, 'tests', 'test_env.py') if DRY else os.path.join(REPO, 'tests')
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', target], capture_output=True, text=True, cwd=REPO)
print(r.stdout[-3000:])
pytest_ok = r.returncode == 0
pytest_summary = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else ''

In [ ]:
# Cell 4: repository audit (no secrets, datasets or checkpoints tracked; .gitignore works)
from eviground.utils.repo_audit import run_audit

audit = run_audit(REPO, history=True)
print(json.dumps(audit, indent=2))

In [ ]:
# Cell 5: verdict, saved to Drive together with the exact package versions
import datetime, pathlib

paths = get_paths().ensure()
out = paths.runs / 'checkpoints'
out.mkdir(parents=True, exist_ok=True)
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
(out / 'requirements.lock.txt').write_text(freeze)

drive_ok = DRY or info['drive_mounted']
datasets_seen = sorted(p.name for p in paths.datasets.iterdir()) if paths.datasets.is_dir() else []
record = {
    'checkpoint': 'CP0',
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'dry_run': DRY,
    'session': info,
    'pytest_ok': pytest_ok,
    'pytest_summary': pytest_summary,
    'audit_ok': audit['ok'],
    'audit': audit,
    'drive_ok': bool(drive_ok),
    'dataset_folders_on_drive': datasets_seen,
    'status': 'PASS' if (pytest_ok and audit['ok'] and drive_ok) else 'FAIL',
}
(out / 'cp0_colab.json').write_text(json.dumps(record, indent=2))
print('CP0 (Colab part):', record['status'])
print('pytest:', pytest_summary)
print('dataset folders on Drive:', datasets_seen)
print('saved to', out / 'cp0_colab.json')
assert record['status'] == 'PASS', 'CP0 failed: see the output of cells 3 and 4'